In [1]:
import pandas as pd 
cleaned_df=pd.read_csv("../outputs/cleaned_retail_orders.csv")
print("Shape of cleaned df is ",cleaned_df.shape)
cleaned_df

Shape of cleaned df is  (8, 9)


,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
0,RT-1001,2026-01-03,Student,Chennai,Learning Kit,2.0,799,10.0,Paid
1,RT-1002,2026-01-03,Fresher,Bengaluru,Course Access,1.0,1499,0.0,Paid
2,RT-1003,2026-01-05,Student,Chennai,Course Access,1.0,1499,0.0,Pending
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3.0,799,5.0,Paid
4,RT-1005,2026-01-09,Fresher,Unknown,Mentor Session,1.0,999,0.0,Failed
5,RT-1008,2026-01-14,Fresher,Bengaluru,Course Access,2.0,1499,0.0,Pending
6,RT-1009,2026-01-16,Student,Chennai,Learning Kit,1.0,799,0.0,Paid
7,RT-1010,2026-01-18,Professional,Delhi,Course Access,2.0,1499,15.0,Refunded


In [2]:
cleaned_df.dtypes

order_id             object
order_date           object
customer_segment     object
city                 object
category             object
quantity            float64
unit_price            int64
discount_pct        float64
payment_status       object
dtype: object

In [3]:
cleaned_df['order_date']=pd.to_datetime(
    cleaned_df['order_date'],format="%Y-%m-%d"
)
cleaned_df.dtypes

order_id                    object
order_date          datetime64[ns]
customer_segment            object
city                        object
category                    object
quantity                   float64
unit_price                   int64
discount_pct               float64
payment_status              object
dtype: object

In [5]:
# Create KPI Dictionary

kpi_dictionary = pd.DataFrame({
    "KPI": [
        "Total Orders",
        "Total Units Sold",
        "Gross Sales",
        "Total Discount",
        "Net Sales",
        "Average Order Value",
        "Average Unit Price",
        "Average Discount %",
        "Paid Order Rate",
        "Sales by Customer Segment",
        "Sales by City",
        "Sales by Category",
        "Daily Sales Trend"
    ],

    "Definition": [
        "Number of unique customer orders",
        "Total number of units sold",
        "Total sales value before discount",
        "Total monetary value of discounts given",
        "Total sales value after discount",
        "Average net sales generated per order",
        "Average selling price per unit",
        "Average percentage discount applied",
        "Percentage of orders with Paid payment status",
        "Net sales grouped by customer segment",
        "Net sales grouped by city",
        "Net sales grouped by product category",
        "Net sales grouped by order date"
    ],

    "Formula": [
        "COUNT(DISTINCT order_id)",
        "SUM(quantity)",
        "SUM(quantity × unit_price)",
        "SUM(gross_amount × discount_pct / 100)",
        "SUM(net_sales)",
        "Net Sales / Total Orders",
        "Net Sales / Total Units Sold",
        "SUM(discount_amount) / SUM(gross_amount) × 100",
        "Paid Orders / Total Orders × 100",
        "SUM(net_sales) GROUP BY customer_segment",
        "SUM(net_sales) GROUP BY city",
        "SUM(net_sales) GROUP BY category",
        "SUM(net_sales) GROUP BY order_date"
    ],

    "Grain": [
        "Order",
        "Order line",
        "Order line",
        "Order line",
        "Order line",
        "Order",
        "Unit",
        "Order line",
        "Order",
        "Customer segment",
        "City",
        "Category",
        "Day"
    ],

    "Filters": [
        "None",
        "None",
        "None",
        "None",
        "None",
        "None",
        "None",
        "None",
        "payment_status = Paid",
        "None",
        "None",
        "None",
        "None"
    ],

    "Owner": [
        "Sales Operations Manager",
        "Sales Operations Manager",
        "Sales Operations Manager",
        "Sales Operations Manager",
        "Sales Operations Manager",
        "Sales Operations Manager",
        "Sales Operations Manager",
        "Sales Operations Manager",
        "Sales Operations Manager",
        "Sales Operations Manager",
        "Sales Operations Manager",
        "Sales Operations Manager",
        "Sales Operations Manager"
    ],

    "Refresh_Cadence": [
        "Daily",
        "Daily",
        "Daily",
        "Daily",
        "Daily",
        "Daily",
        "Daily",
        "Daily",
        "Daily",
        "Daily",
        "Daily",
        "Daily",
        "Daily"
    ]
})

display(kpi_dictionary)

,KPI,Definition,Formula,Grain,Filters,Owner,Refresh_Cadence
0,Total Orders,Number of unique customer orders,COUNT(DISTINCT order_id),Order,None,Sales Operations Manager,Daily
1,Total Units Sold,Total number of units sold,SUM(quantity),Order line,None,Sales Operations Manager,Daily
2,Gross Sales,Total sales value before discount,SUM(quantity × unit_price),Order line,None,Sales Operations Manager,Daily
3,Total Discount,Total monetary value of discounts given,SUM(gross_amount × discount_pct / 100),Order line,None,Sales Operations Manager,Daily
4,Net Sales,Total sales value after discount,SUM(net_sales),Order line,None,Sales Operations Manager,Daily
5,Average Order Value,Average net sales generated per order,Net Sales / Total Orders,Order,None,Sales Operations Manager,Daily
6,Average Unit Price,Average selling price per unit,Net Sales / Total Units Sold,Unit,None,Sales Operations Manager,Daily
7,Average Discount %,Average percentage discount applied,SUM(discount_amount) / SUM(gross_amount) × 100,Order line,None,Sales Operations Manager,Daily
8,Paid Order Rate,Percentage of orders with Paid payment status,Paid Orders / Total Orders × 100,Order,payment_status = Paid,Sales Operations Manager,Daily
9,Sales by Customer Segment,Net sales grouped by customer segment,SUM(net_sales) GROUP BY customer_segment,Customer segment,None,Sales Operations Manager,Daily


In [6]:
kpi_dictionary.to_csv(
    "../outputs/kpi_dictionary.csv",
    index=False
)

print("KPI dictionary saved successfully.")

KPI dictionary saved successfully.


In [7]:
# KPI 1: Total Orders
total_orders = cleaned_df["order_id"].nunique()
print("Total Orders:", total_orders)

Total Orders: 8


In [8]:
# KPI 2: Total Units Sold
total_units_sold = cleaned_df["quantity"].sum()
print("Total Units Sold:", total_units_sold)

Total Units Sold: 13.0


In [9]:
# KPI 3: Gross Sales
cleaned_df["gross_amount"] = (
    cleaned_df["quantity"] * cleaned_df["unit_price"]
)
gross_sales = cleaned_df["gross_amount"].sum()
print("Gross Sales:", gross_sales)

Gross Sales: 14787.0


In [11]:
# KPI 4: Total Discount
cleaned_df["discount_amount"] = (
    cleaned_df["gross_amount"] *
    cleaned_df["discount_pct"] / 100
)
total_discount = cleaned_df["discount_amount"].sum()
print("Total Discount:", round(total_discount,2))

Total Discount: 729.35


In [13]:
# KPI 5: Net Sales
cleaned_df["net_sales"] = (
    cleaned_df["gross_amount"] -
    cleaned_df["discount_amount"]
)
net_sales = cleaned_df["net_sales"].sum()
print("Net Sales: ", round(net_sales, 2))

Net Sales:  14057.65


In [14]:
# KPI 6: Average Order Value
average_order_value = net_sales / total_orders
print("Average Order Value: ", round(average_order_value, 2))

Average Order Value:  1757.21


In [16]:
# KPI 7: Average Unit Price
average_unit_price = cleaned_df["unit_price"].mean()
print("Average Unit Price: ", round(average_unit_price, 2))

Average Unit Price:  1174.0


In [17]:
# KPI 8: Average Discount %
average_discount_pct = (
    total_discount / gross_sales
) * 100
print("Average Discount %:", round(average_discount_pct, 2), "%")

Average Discount %: 4.93 %


In [18]:
# KPI 9: Paid Order Rate
paid_orders = (
    cleaned_df["payment_status"] == "Paid"
).sum()
paid_order_rate = (
    paid_orders / total_orders
) * 100
print("Paid Orders:", paid_orders)
print("Paid Order Rate:", round(paid_order_rate, 2), "%")

Paid Orders: 4
Paid Order Rate: 50.0 %


In [19]:
# KPI 10: Sales by Customer Segment
sales_by_segment = (
    cleaned_df
    .groupby("customer_segment")["net_sales"]
    .sum()
    .reset_index()
    .sort_values("net_sales", ascending=False)
)
sales_by_segment["net_sales"] = sales_by_segment["net_sales"].round(2)
print(sales_by_segment)

  customer_segment  net_sales
0          Fresher    5496.00
1     Professional    4825.45
2          Student    3736.20


In [20]:
# KPI 11: Sales by City
sales_by_city = (
    cleaned_df
    .groupby("city")["net_sales"]
    .sum()
    .reset_index()
    .sort_values("net_sales", ascending=False)
)
sales_by_city["net_sales"] = sales_by_city["net_sales"].round(2)
print(sales_by_city)

        city  net_sales
0  Bengaluru    4497.00
1    Chennai    3736.20
2      Delhi    2548.30
3  Hyderabad    2277.15
4    Unknown     999.00


In [21]:
# KPI 12: Sales by Category
sales_by_category = (
    cleaned_df
    .groupby("category")["net_sales"]
    .sum()
    .reset_index()
    .sort_values("net_sales", ascending=False)
)
sales_by_category["net_sales"] = sales_by_category["net_sales"].round(2)
print(sales_by_category)

         category  net_sales
0   Course Access    8544.30
1    Learning Kit    4514.35
2  Mentor Session     999.00


In [22]:
# KPI 13: Daily Sales Trend
daily_sales = (
    cleaned_df
    .groupby("order_date")["net_sales"]
    .sum()
    .reset_index()
    .sort_values("order_date")
)
daily_sales["net_sales"] = daily_sales["net_sales"].round(2)
print(daily_sales)

  order_date  net_sales
0 2026-01-03    2937.20
1 2026-01-05    1499.00
2 2026-01-07    2277.15
3 2026-01-09     999.00
4 2026-01-14    2998.00
5 2026-01-16     799.00
6 2026-01-18    2548.30


In [24]:
# Create final KPI summary for KPIs 1–9
kpi_summary = pd.DataFrame({
    "KPI": [
        "Total Orders",
        "Total Units Sold",
        "Gross Sales",
        "Total Discount",
        "Net Sales",
        "Average Order Value",
        "Average Unit Price",
        "Average Discount %",
        "Paid Order Rate"
    ],

    "Value": [
        total_orders,
        total_units_sold,
        gross_sales,
        total_discount,
        net_sales,
        average_order_value,
        average_unit_price,
        average_discount_pct,
        paid_order_rate
    ]
})

# Round monetary and percentage values
kpi_summary["Value"] = kpi_summary["Value"].round(2)
display(kpi_summary)

# Save KPI summary
kpi_summary.to_csv(
    "../outputs/kpi_summary.csv",index=False
)
print("KPI summary saved successfully.")

,KPI,Value
0,Total Orders,8.00
1,Total Units Sold,13.00
2,Gross Sales,14787.00
3,Total Discount,729.35
4,Net Sales,14057.65
5,Average Order Value,1757.21
6,Average Unit Price,1174.00
7,Average Discount %,4.93
8,Paid Order Rate,50.00


KPI summary saved successfully.


In [25]:
# Save grouped KPI results
sales_by_segment.to_csv(
    "../outputs/sales_by_customer_segment.csv",index=False
)
sales_by_city.to_csv(
    "../outputs/sales_by_city.csv",index=False
)
sales_by_category.to_csv(
    "../outputs/sales_by_category.csv",index=False
)
daily_sales.to_csv(
    "../outputs/daily_sales_trend.csv",index=False
)
print("All grouped KPI results saved successfully.")

All grouped KPI results saved successfully.
